In [9]:
def compare_chains(chain1, chain2, mismatch_penalty=1, donor_penalty=3, acceptor_penalty=3):
    """
    Compare two chains of intervals and compute matches, mismatches, and penalties.
    
    Args:
        chain1: List of tuples [(start, end), ...] for the first chain.
        chain2: List of tuples [(start, end), ...] for the second chain.
        mismatch_penalty: Penalty for mismatched positions (general).
        donor_penalty: Penalty for donor site mismatches.
        acceptor_penalty: Penalty for acceptor site mismatches.
    
    Returns:
        A dictionary with keys:
        - matches: Total number of matched positions.
        - mismatches: Total number of mismatched positions.
        - donor_mismatches: Total number of donor site mismatches.
        - acceptor_mismatches: Total number of acceptor site mismatches.
        - total_penalty: Sum of all mismatch penalties.
    """
    def flatten_chain(chain):
        """Flatten intervals into a set of positions."""
        return set(pos for start, end in chain for pos in range(start, end + 1))
    
    def get_donor_acceptor_sites(chain):
        """Get donor (end) and acceptor (start) sites separately."""
        donors = {interval[1] for interval in chain[:-1]}  # End of exons except last
        acceptors = {interval[0] for interval in chain[1:]}  # Start of exons except first
        return donors, acceptors
    
    # Flatten the chains into sets of positions
    positions1 = flatten_chain(chain1)
    positions2 = flatten_chain(chain2)
    
    # Compute matches and mismatches
    matches = positions1 & positions2
    mismatches = (positions1 - positions2) | (positions2 - positions1)
    
    # Identify donor and acceptor mismatches
    donors1, acceptors1 = get_donor_acceptor_sites(chain1)
    donors2, acceptors2 = get_donor_acceptor_sites(chain2)
    
    donor_mismatches = (donors1 - donors2) | (donors2 - donors1)
    acceptor_mismatches = (acceptors1 - acceptors2) | (acceptors2 - acceptors1)
    
    # Calculate penalties
    match_count = len(matches)
    mismatch_count = len(mismatches)
    donor_mismatch_count = len(donor_mismatches)
    acceptor_mismatch_count = len(acceptor_mismatches)
    
    total_penalty = (
        mismatch_count * mismatch_penalty +
        donor_mismatch_count * donor_penalty +
        acceptor_mismatch_count * acceptor_penalty
    )
    
    return {
        "matches": match_count,
        "mismatches": mismatch_count,
        "donor_mismatches": donor_mismatch_count,
        "acceptor_mismatches": acceptor_mismatch_count,
        "total_penalty": total_penalty,
    }


In [ ]:
chain1 = [(3, 5), (7, 9)]
chain2 = [(4, 5), (9, 10)]

result = compare_chains(chain1, chain2)
print(result)

{'matches': 4, 'mismatches': 4, 'donor_mismatches': 2, 'acceptor_mismatches': 2, 'total_penalty': 16}


In [ ]:
import re

def parse_cigar_into_tuples(cigar_string: str):
    """Convert CIGAR string to list of tuples"""
    return [(int(length), op) for length, op in re.findall(r'(\d+)(\D)', cigar_string)]

def build_cigar_from_tuples(ops):
    "les to CIGAR string"""
    return ''.join(f"{length}{op}" for length, op in ops)

In [15]:
def shorten_cigar_inplace(cigar, shorten_length:int, from_end:bool=False, offset:int=0):
    """
    Shorten CIGAR string in place by n bases by changing ops to I. Can be done at the start or end of the CIGAR string.
    I on M = I, I on D = 0, I on I = skip to next non-I op.
    """
    if from_end:
        cigar.reverse()

    idx = 0
    remaining_length = shorten_length
    cur_offset = 0

    while remaining_length > 0 and idx < len(cigar):
        oplen, op = cigar[idx]
        
        if cur_offset<offset:
            # skip until offset is reached
            if oplen > offset - cur_offset: # split current operation and we are done
                cigar.insert(idx, (offset - cur_offset, op))
                oplen -= offset - cur_offset
                # update the current operation
                cigar[idx+1] = (oplen, op)
                cur_offset = offset
                idx += 1
            else: # consume the whole operation
                cur_offset += oplen
                idx += 1
                continue

        if op == 'M':  # consume M and replace with I
            ilen = min(oplen, remaining_length)
            cigar[idx] = (ilen, 'I')
            remaining_length -= ilen
            if oplen > ilen:
                cigar.insert(idx + 1, (oplen - ilen, 'M'))
            idx += 1
        elif op == 'D':  # negate D
            ilen = min(oplen, remaining_length)
            remaining_length -= ilen
            if oplen > ilen:
                cigar[idx] = (oplen - ilen, 'D')
                idx += 1
            else:
                del cigar[idx]  # remove this operation if fully consumed
        elif op == 'I':  # skip I
            idx += 1
        else:
            raise ValueError(f"Unsupported CIGAR operation: {op}")

    assert remaining_length == 0, "Remaining length should be zero after processing all ops"

    if from_end:
        cigar.reverse()
        
        
def elongate_cigar_inplace(cigar, elongate_length:int, from_end:bool=False, offset:int=0):
    """
    Elongate CIGAR string by n bases by changing ops to D. Can be done at the start or end of the CIGAR string.
    Modifies the CIGAR list in-place.
    D on I = DI, D on M = DM, D on D = 2D
    """
    # Reverse cigar for processing from the end if needed
    if from_end:
        cigar.reverse()

    remaining_length = elongate_length
    idx = 0
    cur_offset = 0

    while remaining_length > 0 and idx < len(cigar):
        oplen, op = cigar[idx]
        
        if cur_offset<offset:
            # skip until offset is reached
            if oplen > offset - cur_offset: # split current operation and we are done
                cigar.insert(idx, (offset - cur_offset, op))
                oplen -= offset - cur_offset
                # update the current operation
                cigar[idx+1] = (oplen, op)
                cur_offset = offset
                idx += 1
            else: # consume the whole operation
                cur_offset += oplen
                idx += 1
                continue

        if op == 'M':  # Add D to M
            cigar.insert(idx, (remaining_length, 'D'))
            remaining_length = 0
            idx += 2  # Skip the newly inserted D and the current M
        elif op == 'D':  # Add to existing D
            cigar[idx] = (oplen + remaining_length, 'D')
            remaining_length = 0
            idx += 1
        elif op == 'I':  # Skip I
            idx += 1
        else:
            raise ValueError(f"Unsupported CIGAR operation: {op}")

    # If remaining_length is non-zero, it means we need to add a new D at the end
    if remaining_length > 0:
        cigar.append((remaining_length, 'D'))

    # Reverse the cigar back if processed from the end
    if from_end:
        cigar.reverse()

In [17]:
cigar = "11M"
ops = parse_cigar_into_tuples(cigar)
shorten_cigar_inplace(ops, 1, from_end=False,offset=2)
new_cigar = build_cigar_from_tuples(ops)
print(new_cigar)

ops = parse_cigar_into_tuples(cigar)
elongate_cigar_inplace(ops, 1, from_end=False,offset=1)
new_cigar = build_cigar_from_tuples(ops)
print(new_cigar)


cigar = "5D11M"
ops = parse_cigar_into_tuples(cigar)
shorten_cigar_inplace(ops, 1, from_end=False)
new_cigar = build_cigar_from_tuples(ops)
print(new_cigar)

ops = parse_cigar_into_tuples(cigar)
elongate_cigar_inplace(ops, 1, from_end=False)
new_cigar = build_cigar_from_tuples(ops)
print(new_cigar)


cigar = "5I11M"
ops = parse_cigar_into_tuples(cigar)
shorten_cigar_inplace(ops, 1, from_end=False)
new_cigar = build_cigar_from_tuples(ops)
print(new_cigar)

ops = parse_cigar_into_tuples(cigar)
elongate_cigar_inplace(ops, 1, from_end=False)
new_cigar = build_cigar_from_tuples(ops)
print(new_cigar)

2M1I8M
1M1D10M
4D11M
6D11M
5I1I10M
5I1D11M
